In [1]:
import json
from pathlib import Path
import pandas as pd
import torch
import numpy as np

DATA = Path("../data/processed")

df_train     = pd.read_parquet(DATA / "df_train.parquet")
df_cancelled = pd.read_parquet(DATA / "df_cancelled.parquet")   

with open(DATA / "station_to_id.json") as f:
    station_to_id = json.load(f)

edge_index = torch.load(DATA / "edge_index.pt")

print(f"df_train:     {len(df_train):>7,} rows")
print(f"df_cancelled: {len(df_cancelled):>7,} rows")
print(f"stations:     {len(station_to_id):>7,}")
print(f"edge_index:   {tuple(edge_index.shape)}")

df_train:     944,682 rows
df_cancelled:  55,318 rows
stations:         143
edge_index:   (2, 1163)


In [2]:
df_train.head(20)

,station_name,xml_station_name,eva,train_name,final_destination_station,delay_in_min,time,is_canceled,train_type,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id
0,Dresden Hbf,Dresden Hbf,08010085,S 1,Meißen Triebischtal,0.000000,2024-07-01 00:00:00,False,S,-2271920085331621501,15,2024-06-30 23:58:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,-2271920085331621501-2406302315-15
1,NaN,"Hauptbahnhof, Saarbrücken",0836075,STB 1,"Riegelsberg Süd, Riegelsberg",0.000000,2024-07-01 00:00:00,False,STB,-788955727001224364,9,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,-788955727001224364-2406302347-9
2,Nürnberg Hbf,Nürnberg Hbf,08000284,Bus EV,Nürnberg Hbf,0.000000,2024-07-01 00:00:00,False,Bus,-2337914853331985387,2,2024-07-01 00:00:00,2024-07-01 00:00:00,NaT,NaT,-2337914853331985387-2406302300-2
3,Hamburg Hbf,Hamburg Hbf (S-Bahn),08098549,S 5,Hamburg Elbgaustraße,0.000000,2024-07-01 00:00:00,False,S,1695252692593883835,7,2024-06-30 23:59:00,2024-06-30 23:59:00,2024-07-01 00:00:00,2024-07-01 00:00:00,1695252692593883835-2406302341-7
4,Bielefeld Hbf,Bielefeld Hbf,08000036,RE 6,Bielefeld Hbf,0.000000,2024-07-01 00:00:00,False,RE,2444314634072177884,23,2024-07-01 00:01:00,2024-07-01 00:00:00,NaT,NaT,2444314634072177884-2406302048-23
5,Düsseldorf Hbf,Düsseldorf Hbf,08000085,S 6,Düsseldorf-Rath Mitte,0.000000,2024-07-01 00:00:00,False,S,7866289327576932582,21,2024-06-30 23:58:00,2024-06-30 23:58:00,2024-07-01 00:00:00,2024-07-01 00:00:00,7866289327576932582-2406302301-21
6,Singen (Hohentwiel),Singen(Hohentwiel),08000073,SBB S6,Engen,0.000000,2024-07-01 00:00:00,False,SBB,8357206333431226217,12,2024-06-30 23:56:00,2024-06-30 23:56:00,2024-07-01 00:00:00,2024-07-01 00:00:00,8357206333431226217-2406302323-12
7,NaN,"ZOB/Hauptbahnhof, Pforzheim",0940370,Bus S6 (S,"Bahnhof, Bad Wildbad",0.000000,2024-07-01 00:00:00,False,Bus,-6129702905591104469,1,NaT,NaT,2024-07-01 00:00:00,2024-07-01 00:00:00,-6129702905591104469-2407010000-1
8,Köln Hbf,Köln Hbf,08000207,S 12,Köln-Ehrenfeld,0.000000,2024-07-01 00:00:00,False,S,-1291355389794506596,13,2024-06-30 23:59:00,2024-06-30 23:58:00,2024-07-01 00:00:00,2024-07-01 00:00:00,-1291355389794506596-2406302318-13
9,Frankfurt (Main) Süd,Frankfurt(Main)Süd,08002041,HLB RB58,Hanau Hbf,0.000000,2024-07-01 00:01:00,False,HLB,-5012084765708061335,4,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:01:00,2024-07-01 00:01:00,-5012084765708061335-2406302333-4


Now I will refactor the data so that we have node features (from our dataset we will get per station features)

In 1 million data we have data worth of 15 days. We will take a snapshot of every hour for 15 days, so we will have 15 x 24 snapshots. Each one is will be shaped (143, 9). The 9 will be the features we will extract from the dataset. (More to it later)

In [3]:
display(df_train)

,station_name,xml_station_name,eva,train_name,final_destination_station,delay_in_min,time,is_canceled,train_type,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id
0,Dresden Hbf,Dresden Hbf,08010085,S 1,Meißen Triebischtal,0.000000,2024-07-01 00:00:00,False,S,-2271920085331621501,15,2024-06-30 23:58:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,-2271920085331621501-2406302315-15
1,NaN,"Hauptbahnhof, Saarbrücken",0836075,STB 1,"Riegelsberg Süd, Riegelsberg",0.000000,2024-07-01 00:00:00,False,STB,-788955727001224364,9,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,-788955727001224364-2406302347-9
2,Nürnberg Hbf,Nürnberg Hbf,08000284,Bus EV,Nürnberg Hbf,0.000000,2024-07-01 00:00:00,False,Bus,-2337914853331985387,2,2024-07-01 00:00:00,2024-07-01 00:00:00,NaT,NaT,-2337914853331985387-2406302300-2
3,Hamburg Hbf,Hamburg Hbf (S-Bahn),08098549,S 5,Hamburg Elbgaustraße,0.000000,2024-07-01 00:00:00,False,S,1695252692593883835,7,2024-06-30 23:59:00,2024-06-30 23:59:00,2024-07-01 00:00:00,2024-07-01 00:00:00,1695252692593883835-2406302341-7
4,Bielefeld Hbf,Bielefeld Hbf,08000036,RE 6,Bielefeld Hbf,0.000000,2024-07-01 00:00:00,False,RE,2444314634072177884,23,2024-07-01 00:01:00,2024-07-01 00:00:00,NaT,NaT,2444314634072177884-2406302048-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,Berlin Zoologischer Garten,Berlin Zoologischer Garten (S),08089046,S 9,Berlin-Spandau (S),0.693147,2024-07-15 20:47:00,False,S,3377647720781430757,21,2024-07-15 20:45:00,2024-07-15 20:46:00,2024-07-15 20:46:00,2024-07-15 20:47:00,3377647720781430757-2407151949-21
999996,Essen Hbf,Essen Hbf,08000098,RE 42,Krefeld Hbf,1.098612,2024-07-15 20:47:00,False,RE,-6075476340840051066,14,2024-07-15 20:43:00,2024-07-15 20:44:00,2024-07-15 20:45:00,2024-07-15 20:47:00,-6075476340840051066-2407151936-14
999997,Chemnitz Hbf,Chemnitz Hbf,08010184,RB 81,Chemnitz Hbf,0.000000,2024-07-15 20:47:00,False,RB,-1660650733487052009,17,2024-07-15 20:47:00,2024-07-15 20:47:00,NaT,NaT,-1660650733487052009-2407151938-17
999998,Hamburg Hbf,Hamburg Hbf,08002549,ME RE4,Hamburg Hbf,1.609438,2024-07-15 20:47:00,False,ME,-657578417068350480,6,2024-07-15 20:43:00,2024-07-15 20:47:00,NaT,NaT,-657578417068350480-2407151933-6


In [4]:
df_train["is_canceled"].sum()

np.int64(0)

In [5]:
df_train["station_id"] = df_train["xml_station_name"].map(station_to_id)

start_time = df_train["time"].min().floor("h")
end_time = df_train["time"].max().ceil("h")
hours = pd.date_range(start=start_time, end=end_time, freq="h")

# make a numpy array with nan values
# after we refactor the matrix, if there is a nan value left, 
# we can say there is a mistake
avg_delay_1h = np.full((len(hours), len(station_to_id)), np.nan)

for i, h in enumerate(hours):
    rides_in_h = df_train[ (df_train["time"] >= h - pd.Timedelta("1h")) 
                        & (df_train["time"] < h)]
    means = rides_in_h.groupby("station_id")["delay_in_min"].mean()
    avg_delay_1h[i, means.index] = means.values


In [6]:
avg_delay_6h = np.full((len(hours), len(station_to_id)), np.nan)
for i, t in enumerate(hours):
    window = df_train[(df_train["time"] >= t - pd.Timedelta("6h")) &
                   (df_train["time"] <  t)]
    means = window.groupby("station_id")["delay_in_min"].mean()
    avg_delay_6h[i, means.index] = means.values

avg_delay_24h = np.full((len(hours), len(station_to_id)), np.nan)
for i, t in enumerate(hours):
    window = df_train[(df_train["time"] >= t - pd.Timedelta("24h")) &
                   (df_train["time"] <  t)]
    means = window.groupby("station_id")["delay_in_min"].mean()
    avg_delay_24h[i, means.index] = means.values

In [7]:
number_of_rides = np.full((len(hours), len(station_to_id)), 0, dtype=int)

for i, h in enumerate(hours):
    rides_in_h = df_train[ (df_train["time"] >= h - pd.Timedelta("1h")) 
                        & (df_train["time"] < h)]
    sum_of_trains = rides_in_h.groupby("station_id").size()
    number_of_rides[i, sum_of_trains.index] = sum_of_trains.values

In [8]:
hour_sin = np.full((len(hours), len(station_to_id)), 0.0)
hour_cos = np.full((len(hours), len(station_to_id)), 0.0)

for i, t in enumerate(hours):
    hour_sin[i, :] = np.sin(2 * np.pi * t.hour / 24)
    hour_cos[i, :] = np.cos(2 * np.pi * t.hour / 24)
    
day_sin = np.full((len(hours), len(station_to_id)), 0.0)
day_cos = np.full((len(hours), len(station_to_id)), 0.0)

for i, t in enumerate(hours):
    day_sin[i, :] = np.sin(2 * np.pi * t.day_of_week / 7)
    day_cos[i, :] = np.cos(2 * np.pi * t.day_of_week / 7)
    

In [9]:
x_snapshots = np.stack([
    avg_delay_1h, avg_delay_6h, avg_delay_24h,
    number_of_rides,
    hour_sin, hour_cos, day_sin, day_cos], axis=-1)

print(x_snapshots.shape) 

(358, 143, 8)


In [10]:
x_snapshots = np.nan_to_num(x_snapshots, nan=0.0) 

In [11]:
# Each ride uses the snapshot at the start of its hour
ride_snapshot_time = df_train["time"].dt.floor("h")
# ride_snapshot_time looks like this:
# 0    date1 00:00
# 1    date2 01:00
# 2    date3 01:00 
# ...

# Building a dictionary to see which hour corresponds to which snapshot
hours_to_snapshot_id = {t: i for i, t in enumerate(hours)}

# We are now getting an array which contains the the index of the snapshot 
# we need to use for the ride
ride_to_snapshot_id = ride_snapshot_time.map(hours_to_snapshot_id).to_numpy()
# ride_to_snapshot_id looked like this when we mapped 
# 0    0
# 1    1
# 2    1
# ...
# but with to_numpy() we made it to an array of [0, 1, 1, ...]
# So it means: for the ride 0 in df_train, use the snapshot index (array[0]) 

# Get the stations used in rides
ride_station_id = df_train["station_id"].to_numpy()

ride_targets = df_train["delay_in_min"].to_numpy()

After we get the station embeddings with GCN, we will need a MLP (Plain feed-forward network) to add ride specific features to finalize the prediction. 

(With the GCN we will get a 32-dim vector but we want to predict "how much a ride might get delayed" so we need a number.)

In [12]:
train_type_encoded = pd.get_dummies(df_train["train_type"])

We used "pandas.get_dummies" because they are categories. If we assigned integers like "IC=0, ICE=1, RE=2" that would mean that ICE is closer/similar to RE than IC. So we used one hot encoding to specify the train types. 

In [13]:
station_num_normalized = (df_train["train_line_station_num"] / df_train["train_line_station_num"].max())

In [14]:
ride_features = pd.concat([train_type_encoded, station_num_normalized], axis=1).to_numpy()

In [ ]:
import torch
from torch_geometric.nn import GCNConv

class DeutscheBahnGNN(torch.nn.Module):
    
    def __init__(self, node_feat_dim, gcn_hidden, station_emb_dim, ride_feat_dim, mlp_hidden):

        super().__init__()
        self.gcn1 = GCNConv(node_feat_dim, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, station_emb_dim)
        self.head1 = torch.nn.Linear(station_emb_dim + ride_feat_dim, mlp_hidden)
        self.head2 = torch.nn.Linear(mlp_hidden, 1)
        
    def forward(self, x, edge_index, station_ids, ride_features):
        
        h = self.gcn1(x, edge_index)
        h = torch.relu(h)
        h= self.gcn2(h, edge_index)
        
        station_embeddings = h[station_ids]
        combined = torch.cat([station_embeddings, ride_features], axis=1 )
        
        h = self.head1(combined)
        h = torch.relu(h)
        h = self.head2(h)
        
        return h.squeeze(-1)

In [17]:
print(train_type_encoded.dtypes)

AKN    bool
ALX    bool
BRB    bool
Bus    bool
CB     bool
CBG    bool
D      bool
EC     bool
ECE    bool
EN     bool
ENO    bool
ES     bool
EST    bool
EVB    bool
FEX    bool
FLX    bool
HBX    bool
HLB    bool
IC     bool
ICE    bool
IRE    bool
ME     bool
MEX    bool
N      bool
NBE    bool
NJ     bool
NWB    bool
OPB    bool
R      bool
RB     bool
RE     bool
RJ     bool
RJX    bool
RRB    bool
RT     bool
S      bool
SAB    bool
SBB    bool
STB    bool
STN    bool
SVG    bool
SWE    bool
TEL    bool
TGV    bool
TL     bool
TLX    bool
TRI    bool
UEX    bool
VIA    bool
WB     bool
WFB    bool
ag     bool
erx    bool
dtype: object
